# Fix Qwen3-VL FP8 — kernels finegrained-fp8

> `Kernel → Restart Kernel and Run All Cells`

3 strategies tentees dans l'ordre, on s'arrete a la premiere qui marche.

## 0. Variables d'environnement (avant tout import)

In [ ]:
import os, sys

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_KERNELS_TRUST_REMOTE_CODE"] = "1"
os.environ["TRUST_REMOTE_CODE"] = "1"
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")

print("✅ Env posees")

## 1. Config chemins

In [ ]:
import subprocess

# --- Chemin modele ---
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-VL-7B-FP8/main"

# Auto-detection si le chemin n'existe pas
from pathlib import Path
if not Path(MODEL_PATH).is_dir():
    print(f"⚠️  {MODEL_PATH} introuvable, recherche...")
    r = subprocess.run(
        ["find", "/domino/edv/modelhub", "-name", "config.json", "-path", "*Qwen*VL*FP8*"],
        capture_output=True, text=True, timeout=60
    )
    for line in r.stdout.strip().split("\n"):
        if line:
            candidate = str(Path(line).parent)
            print(f"  Trouve : {candidate}")
            MODEL_PATH = candidate
            break
    print(f"  MODEL_PATH = {MODEL_PATH}")

assert Path(MODEL_PATH).is_dir(), f"❌ Dossier modele introuvable: {MODEL_PATH}"
assert (Path(MODEL_PATH) / "config.json").exists(), f"❌ config.json manquant dans {MODEL_PATH}"
print(f"✅ Modele : {MODEL_PATH}")

# --- Chemin kernel Domino ---
KERNEL_DOMINO = "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8"
assert Path(KERNEL_DOMINO).is_dir(), f"❌ Kernel introuvable: {KERNEL_DOMINO}"

# Lister les versions disponibles
versions = sorted([d.name for d in Path(KERNEL_DOMINO).iterdir() if d.is_dir()])
print(f"✅ Kernel : {KERNEL_DOMINO}")
print(f"   Versions : {versions}")

# Trouver v4 (ou la plus recente)
if "v4" in versions:
    KERNEL_V4 = str(Path(KERNEL_DOMINO) / "v4")
elif versions:
    KERNEL_V4 = str(Path(KERNEL_DOMINO) / versions[-1])
    print(f"   ⚠️  v4 non trouvee, utilisation de {versions[-1]}")
else:
    KERNEL_V4 = None
    print("   ❌ Aucune version trouvee")

if KERNEL_V4:
    print(f"   v4 path  : {KERNEL_V4}")
    print(f"   Contenu  : {os.listdir(KERNEL_V4)}")

## Strategie A — Monkey-patch `get_kernel`

On intercepte l'appel `get_kernel("kernels-community/finegrained-fp8", version=4)`
et on retourne directement le module charge depuis le disque Domino.

C'est le fix le plus robuste car il bypass completement le cache HF Hub.

In [ ]:
STRATEGY_A = False

try:
    import importlib, types
    import kernels
    from kernels import get_kernel as _original_get_kernel

    # Trouver le .so dans build/
    build_dir = Path(KERNEL_V4) / "build"
    so_files = list(build_dir.glob("**/*.so")) if build_dir.exists() else []
    print(f"Fichiers .so trouves dans v4/build/ : {len(so_files)}")
    for f in so_files[:10]:
        print(f"  {f}")

    if not so_files:
        # Peut-etre que les kernels sont en .py ou en .cu
        all_files = list(build_dir.rglob("*")) if build_dir.exists() else []
        print(f"Tous les fichiers dans build/ : {len(all_files)}")
        for f in all_files[:20]:
            print(f"  {f.relative_to(build_dir)}")

    # Tenter de charger le kernel via importlib depuis le path
    def patched_get_kernel(repo_id, *args, **kwargs):
        if "finegrained" in repo_id:
            print(f"  [PATCH] get_kernel({repo_id!r}) → chargement local depuis {KERNEL_V4}")
            # Ajouter build/ au sys.path
            build_path = str(Path(KERNEL_V4) / "build")
            if build_path not in sys.path:
                sys.path.insert(0, build_path)
            # Essayer d'importer le module
            try:
                import finegrained_fp8
                return finegrained_fp8
            except ImportError:
                # Essayer avec un import dynamique du .so
                if so_files:
                    spec = importlib.util.spec_from_file_location("finegrained_fp8", str(so_files[0]))
                    mod = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(mod)
                    return mod
                raise
        return _original_get_kernel(repo_id, *args, **kwargs)

    kernels.get_kernel = patched_get_kernel
    # Aussi patcher dans transformers si il a sa propre reference
    try:
        import transformers.integrations.hub_kernels as hk
        if hasattr(hk, "get_kernel_hub"):
            _orig_hk = hk.get_kernel_hub
            def patched_hk(*args, **kwargs):
                try:
                    return _orig_hk(*args, **kwargs)
                except Exception:
                    return patched_get_kernel(*args, **kwargs)
            hk.get_kernel_hub = patched_hk
            print("  Aussi patche transformers.integrations.hub_kernels")
    except Exception:
        pass

    print("✅ Strategie A : monkey-patch installe")
    STRATEGY_A = True

except Exception as e:
    print(f"❌ Strategie A echouee : {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

## Strategie B — Creer la structure de cache HF Hub manuellement

La lib `kernels` utilise `huggingface_hub` en interne.
Le cache HF attend cette arborescence :

```
~/.cache/huggingface/hub/models--kernels-community--finegrained-fp8/
├── refs/
│   ├── main → <fake-hash>
│   └── v4   → <fake-hash-v4>
├── snapshots/
│   ├── <fake-hash>/    → symlink vers .../main/
│   └── <fake-hash-v4>/ → symlink vers .../v4/
```

On la cree avec des faux hashes et des symlinks.

In [ ]:
STRATEGY_B = False

try:
    import hashlib

    cache_base = Path.home() / ".cache" / "huggingface" / "hub" / "models--kernels-community--finegrained-fp8"
    refs_dir = cache_base / "refs"
    snap_dir = cache_base / "snapshots"

    # Creer les dossiers
    refs_dir.mkdir(parents=True, exist_ok=True)
    snap_dir.mkdir(parents=True, exist_ok=True)

    # Pour chaque version sur Domino
    kernel_base = Path(KERNEL_DOMINO)
    for version_name in ["main", "v4"]:
        version_path = kernel_base / version_name
        if not version_path.exists():
            print(f"  {version_name}/ n'existe pas, skip")
            continue

        # Generer un hash fake mais stable
        fake_hash = hashlib.sha256(f"domino-local-{version_name}".encode()).hexdigest()[:40]

        # Creer le fichier refs/<version> contenant le hash
        ref_file = refs_dir / version_name
        ref_file.write_text(fake_hash)

        # Creer le symlink snapshots/<hash> → chemin Domino
        snap_link = snap_dir / fake_hash
        if snap_link.exists() or snap_link.is_symlink():
            snap_link.unlink()
        snap_link.symlink_to(version_path)

        print(f"  {version_name}: refs/{version_name} → {fake_hash[:12]}... → {version_path}")

    # Aussi creer refs pour les versions numeriques
    for num_ref in ["4"]:
        src = refs_dir / "v4"
        dst = refs_dir / num_ref
        if src.exists() and not dst.exists():
            import shutil
            shutil.copy2(str(src), str(dst))
            print(f"  Copie refs/v4 → refs/{num_ref}")

    # Verifier
    print(f"\n  Arborescence creee :")
    for p in sorted(cache_base.rglob("*")):
        rel = p.relative_to(cache_base)
        if p.is_symlink():
            print(f"    {rel} → {p.resolve()}")
        elif p.is_file():
            print(f"    {rel} ({p.read_text().strip()[:40]})")
        else:
            print(f"    {rel}/")

    print("✅ Strategie B : cache HF Hub cree")
    STRATEGY_B = True

except Exception as e:
    print(f"❌ Strategie B echouee : {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

## Strategie C — Desactiver completement les kernels dans transformers

On monkey-patche la couche d'integration transformers → kernels pour que
tout appel a `get_kernel_hub` retourne None (fallback PyTorch pur).

Plus lent mais garanti de marcher.

In [ ]:
STRATEGY_C = False

try:
    # Methode 1 : patcher hub_kernels dans transformers
    try:
        from transformers.integrations import hub_kernels
        hub_kernels.get_kernel_hub = lambda *a, **kw: None
        print("  Patche transformers.integrations.hub_kernels.get_kernel_hub → None")
    except (ImportError, AttributeError) as e:
        print(f"  hub_kernels non trouvable: {e}")

    # Methode 2 : patcher get_kernel dans le module kernels
    try:
        import kernels
        if not STRATEGY_A:  # ne pas ecraser le patch A si il a marche
            kernels.get_kernel = lambda *a, **kw: None
            print("  Patche kernels.get_kernel → None")
    except ImportError:
        pass

    # Methode 3 : patcher quantizer_compressed_tensors pour ne pas utiliser les kernels
    try:
        from transformers.quantizers import quantizer_compressed_tensors as qct
        if hasattr(qct, "USE_KERNELS"):
            qct.USE_KERNELS = False
            print("  quantizer_compressed_tensors.USE_KERNELS = False")

        # Chercher et patcher la reference au kernel
        import types
        for name in dir(qct):
            obj = getattr(qct, name)
            if isinstance(obj, type) and "Quantizer" in name:
                for method_name in dir(obj):
                    if "kernel" in method_name.lower():
                        print(f"  Trouve {name}.{method_name}")
    except (ImportError, AttributeError) as e:
        print(f"  quantizer patch: {e}")

    print("✅ Strategie C : kernels completement desactivees")
    STRATEGY_C = True

except Exception as e:
    print(f"❌ Strategie C echouee : {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

## Test de chargement du modele

In [ ]:
import time, torch
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f"Strategies actives : A={STRATEGY_A}, B={STRATEGY_B}, C={STRATEGY_C}")
print()

# --- Processor ---
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True,
)
processor.tokenizer.padding_side = "left"
print(f"✅ Processor en {time.time()-t0:.1f}s")

# --- Modele ---
t0 = time.time()
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    model.eval()
    print(f"✅ Modele charge en {time.time()-t0:.1f}s")
    print(f"   dtype  : {next(model.parameters()).dtype}")
    print(f"   device : {next(model.parameters()).device}")
    MODEL_OK = True
except Exception as e:
    print(f"❌ ECHEC chargement: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    MODEL_OK = False

    # Derniere tentative : torch_dtype au lieu de dtype (compat ancienne API)
    if "unexpected keyword" in str(e) or "dtype" in str(e):
        print("\n--- Retry avec torch_dtype= ---")
        try:
            model = AutoModelForImageTextToText.from_pretrained(
                MODEL_PATH,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True,
                low_cpu_mem_usage=True,
                local_files_only=True,
            )
            model.eval()
            print(f"✅ Modele charge avec torch_dtype en {time.time()-t0:.1f}s")
            MODEL_OK = True
        except Exception as e2:
            print(f"❌ ECHEC aussi: {e2}")

## Test inference

In [ ]:
if MODEL_OK:
    from PIL import Image
    import numpy as np

    # Image grise factice
    img = Image.fromarray(np.full((512, 512, 3), 200, dtype=np.uint8))
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": "Decris ce que tu vois."},
    ]}]

    try:
        text_in = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        text_in = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

    DEVICE = next(model.parameters()).device
    inputs = processor(text=[text_in], images=[img], return_tensors="pt").to(DEVICE)
    print(f"Inputs : {inputs['input_ids'].shape} sur {DEVICE}")

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False,
                             pad_token_id=processor.tokenizer.eos_token_id)
    elapsed = time.time() - t0
    generated = out[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(generated, skip_special_tokens=True)
    print(f"✅ Inference OK en {elapsed:.1f}s")
    print(f"   Reponse : {text[:300]}")
else:
    print("❌ Modele non charge — inference impossible")

## Bilan final

In [ ]:
print("=" * 60)
print("BILAN FINAL")
print("=" * 60)
print(f"  Strategie A (monkey-patch get_kernel) : {STRATEGY_A}")
print(f"  Strategie B (cache HF Hub manuel)     : {STRATEGY_B}")
print(f"  Strategie C (desactivation kernels)   : {STRATEGY_C}")
print(f"  Modele charge                         : {MODEL_OK}")
print()
if MODEL_OK:
    print("🎉 Le modele fonctionne !")
    print("   → Copie les cellules 0 + strategies qui ont marche dans ton pipeline V13.")
    print(f"   → Strategies utilisees : ", end="")
    if STRATEGY_A: print("A (monkey-patch)", end=" ")
    if STRATEGY_B: print("B (cache HF)", end=" ")
    if STRATEGY_C: print("C (kernels off)", end=" ")
    print()
else:
    print("❌ Aucune strategie n'a suffi.")
    print("   → Verifie les tracebacks ci-dessus.")
    print("   → Option finale : utiliser le checkpoint BF16 au lieu de FP8.")
    print("   → Cherche : find /domino/edv/modelhub -name 'config.json' -path '*Qwen*VL*' | grep -iv fp8")